In [4]:
# 로더/경로/모델/옵티마이저
from preparation import get_train_val_loaders, get_model_paths
from tqdm.auto import tqdm

import torch
from torchvision.models.detection.ssd import ssd300_vgg16
from torchvision.models import VGG16_Weights

train_data_loader, val_data_loader = get_train_val_loaders()
paths = get_model_paths()

NUM_CLASSES = 3

if torch.backends.mps.is_available() and torch.backends.mps.is_built():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda:0")
else:
    DEVICE = torch.device("cpu")

# bbox 학습 안정화를 위해 backbone만 pretrained 사용
model = ssd300_vgg16(
    weights=None,
    weights_backbone=VGG16_Weights.DEFAULT,
    num_classes=NUM_CLASSES,
).to(DEVICE)

optim = torch.optim.SGD(
    model.parameters(),
    lr=1e-4,
    momentum=0.9,
    weight_decay=5e-4,
)

scheduler = torch.optim.lr_scheduler.StepLR(optim, step_size=2, gamma=0.1)
print("DEVICE:", DEVICE)
print("SAVE PATH:", paths["basic_pth"])

DEVICE: mps
SAVE PATH: /Users/won/dev/00_codeit/0_mission/16_etc_quantization/models/model.pth


In [ ]:
def _to_device_batch(images, targets, device):
    images = [img.to(device) for img in images]
    new_targets = []
    for t in targets:
        # SSD가 실제로 쓰는 키만 넘김
        new_targets.append({
            "boxes": t["boxes"].to(device),
            "labels": t["labels"].to(device),
        })
    return images, new_targets


def train_one_epoch(model, loader, optimizer, device, epoch, num_epochs):
    model.train()
    running = 0.0

    pbar = tqdm(loader, desc=f"[Train] {epoch}/{num_epochs}", leave=False)
    for images, targets in pbar:
        images, targets = _to_device_batch(images, targets, device)

        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())

        if not torch.isfinite(loss):
            continue

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        loss_val = float(loss.item())
        running += loss_val
        pbar.set_postfix(loss=f"{loss_val:.4f}")

    return running / max(len(loader), 1)


@torch.no_grad()
def validate_loss(model, loader, device, epoch, num_epochs):
    # detection 모델은 loss 계산 시 train 모드 필요
    model.train()
    running = 0.0

    pbar = tqdm(loader, desc=f"[Val] {epoch}/{num_epochs}", leave=False)
    for images, targets in pbar:
        images, targets = _to_device_batch(images, targets, device)

        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())
        loss_val = float(loss.item())
        running += loss_val
        pbar.set_postfix(loss=f"{loss_val:.4f}")

    return running / max(len(loader), 1)


num_epochs = 1
best_val = float("inf")

for epoch in range(1, num_epochs + 1):
    train_loss = train_one_epoch(model, train_data_loader, optim, DEVICE, epoch, num_epochs)
    val_loss = validate_loss(model, val_data_loader, DEVICE, epoch, num_epochs)
    scheduler.step()

    print(
        f"[Epoch {epoch}/{num_epochs}] "
        f"train_loss={train_loss:.4f} val_loss={val_loss:.4f} lr={optim.param_groups[0]['lr']:.2e}"
    )

    if val_loss < best_val:
        best_val = val_loss
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "num_classes": NUM_CLASSES,
                "best_val_loss": best_val,
            },
            paths["basic_pth"],
        )
        print(f"  -> best model saved: {paths['basic_pth']} (val_loss={best_val:.4f})")

KeyboardInterrupt: 